# AI Engineering — RAG chatbot> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## ElőfeltételOpenAI API kulcs kell (olcsó: ~$0.01 a teljes notebookra). Állítsd be környezeti változóként:```bash# Linux/Macexport OPENAI_API_KEY=sk-...# Windows PowerShell$env:OPENAI_API_KEY="sk-..."```Ha csak strukturát akarod látni, a valódi hívásokat ki tudod kommentezni.

In [ ]:
%pip install openai chromadb --quiet

In [ ]:
import osos.environ.setdefault('OPENAI_API_KEY', 'sk-REPLACE-ME')from openai import OpenAIclient = OpenAI()print('OK — kliens létrehozva')

## 1. Első chat completion

In [ ]:
response = client.chat.completions.create(    model='gpt-4o-mini',    messages=[        {'role': 'system', 'content': 'Segítőkész magyar asszisztens vagy egy webshopnak.'},        {'role': 'user',   'content': 'Mi a visszaküldési határidő?'}    ],    temperature=0.2,)print(response.choices[0].message.content)

## 2. Dokumentum-corpus — WebShop Pro GYIK

In [ ]:
faq_docs = [    {'id': 'f01', 'text': 'A visszaküldési határidő 14 nap a vásárlástól. A terméknek bontatlan és sértetlen állapotban kell lennie.'},    {'id': 'f02', 'text': 'A kiszállítás Magyarországra 2-3 munkanap, díja 1490 Ft, 15.000 Ft feletti rendelésnél ingyenes.'},    {'id': 'f03', 'text': 'Fizetési módok: bankkártya, PayPal, utánvét (+500 Ft), átutalás. Részletfizetés CIB Bank partnerünkkel.'},    {'id': 'f04', 'text': 'A garanciális jogokat a gyártó 24 hónap garancia keretében biztosítja. Meghibásodás esetén nyugtával forduljon hozzánk.'},    {'id': 'f05', 'text': 'A rendeléseket H-P 8-17 között dolgozzuk fel. A hétvégén érkezett rendelések hétfőn kerülnek sorra.'},    {'id': 'f06', 'text': 'A regisztrált ügyfelek minden rendelés után 5% hűségpontot kapnak, ami a következő vásárlásnál levásárolható.'},]print(f'{len(faq_docs)} dokumentum')

## 3. Embedding — szöveg → vektor

In [ ]:
texts = [d['text'] for d in faq_docs]emb_response = client.embeddings.create(model='text-embedding-3-small', input=texts)for doc, emb in zip(faq_docs, emb_response.data):    doc['embedding'] = emb.embeddingprint(f'Embedding dimenziók: {len(faq_docs[0]["embedding"])}')print(f'Első 5 érték: {faq_docs[0]["embedding"][:5]}')

## 4. ChromaDB vektoros tárolás

In [ ]:
import chromadbchroma = chromadb.Client()col = chroma.get_or_create_collection(name='webshop-faq')col.add(    ids=[d['id'] for d in faq_docs],    documents=[d['text'] for d in faq_docs],    embeddings=[d['embedding'] for d in faq_docs],)print(f'ChromaDB méret: {col.count()} dokumentum')

## 5. Szemantikus keresés — retrieval

In [ ]:
def retrieve(query: str, k: int = 3) -> list[str]:    q_emb = client.embeddings.create(model='text-embedding-3-small', input=[query]).data[0].embedding    hit = col.query(query_embeddings=[q_emb], n_results=k)    return hit['documents'][0]question = 'Mennyi ideig küldhetek vissza egy terméket?'hits = retrieve(question)print(f'Kérdés: {question}\n')for i, h in enumerate(hits, 1):    print(f'[{i}] {h}')

## 6. RAG — Retrieval-Augmented Generation

In [ ]:
def rag_answer(question: str) -> str:    context_docs = retrieve(question, k=3)    context = '\n'.join(f'- {doc}' for doc in context_docs)    response = client.chat.completions.create(        model='gpt-4o-mini',        messages=[            {'role': 'system', 'content':                'Segítőkész magyar asszisztens vagy. Kizárólag a kontextusból válaszolj. '                'Ha nincs benne a válasz, mondd meg, hogy nem tudod.'},            {'role': 'user', 'content':                f'Kontextus:\n{context}\n\nKérdés: {question}'}        ],        temperature=0.1,    )    return response.choices[0].message.contentfor q in ['Mennyibe kerül a kiszállítás?', 'Van-e részletfizetés?', 'Szerviz elérhető?']:    print(f'\n❓ {q}\n💬 {rag_answer(q)}')

## 7. Structured output — JSON mód

In [ ]:
from pydantic import BaseModelclass FAQIntent(BaseModel):    category: str     # 'delivery', 'payment', 'warranty', 'other'    confidence: float # 0-1    summary: str      # rövid összefoglalóresponse = client.beta.chat.completions.parse(    model='gpt-4o-mini',    messages=[        {'role': 'system', 'content': 'Kategorizáld a felhasználó kérdését.'},        {'role': 'user',   'content': 'Mikor kapom meg a csomagot?'}    ],    response_format=FAQIntent,)intent = response.choices[0].message.parsedprint(intent.model_dump_json(indent=2))

## Következő lépések- Térj vissza a [web-alapú kurzushoz](ai-engineering/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*